# CMIP6 Standardized Precipitation Index

This notebook is a simple Colab reference for the CMIP6 SPI pipeline. It follows the same sequence as the script: open the preprocessed precipitation data, convert the daily flux, aggregate it by month, fit the distribution with the historical calibration data, calculate SPI for the selected experiment, add metadata, and save a NetCDF file.

In [ ]:
!pip -q install xarray xclim zarr dask numcodecs cftime netCDF4

## Imports

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import xarray as xr
from xclim.indices import standardized_precipitation_index
from xclim.indices.stats import standardized_index_fit_params

## Configuration

Set these values for the experiment being processed. The calibration file is normally the preprocessed historical file. For a historical run, it can be the same file as the input file.

In [ ]:
BASE_PATH = Path("/content")
CALIBRATION_FILE = BASE_PATH / "path/to/preprocessed_historical_precipitation.zarr"
INPUT_FILE = BASE_PATH / "path/to/preprocessed_experiment_precipitation.zarr"
MODEL = "ACCESS-CM2"
EXPERIMENT = "ssp245"
MEMBER = "r1i1p1f1"
GRID = "gn"

PRECIPITATION_VARIABLE = "pr"
TIME_DIMENSION = "time"
LATITUDE_DIMENSION = "lat"
LONGITUDE_DIMENSION = "lon"
SPI_SCALE_MONTHS = 1
SPI_DISTRIBUTION = "gamma"
SPI_METHOD = "APP"
SPI_FLOC = 0
CALIBRATION_START = "1961-01-01"
CALIBRATION_END = "1990-12-31"
APPLICATION_START = "2015-01-01"
APPLICATION_END = "2050-12-31"
OUTPUT_FILE = BASE_PATH / f"spi{SPI_SCALE_MONTHS}_{MODEL}_{EXPERIMENT}_{MEMBER}_{GRID}_{APPLICATION_START}_{APPLICATION_END}.nc"

## Helper functions

In [ ]:
def open_precipitation(path: Path) -> xr.Dataset:
    if path.suffix == ".zarr" or path.is_dir():
        return xr.open_zarr(path)
    return xr.open_dataset(path)


def to_monthly_precipitation(dataset: xr.Dataset) -> xr.DataArray:
    units = dataset[PRECIPITATION_VARIABLE].attrs.get("units", "")
    if units not in {"kg m-2 s-1", "kg m**-2 s**-1", "mm s-1"}:
        raise ValueError("Expected a daily precipitation flux")
    precipitation = dataset[PRECIPITATION_VARIABLE].rename(
        {TIME_DIMENSION: "time", LATITUDE_DIMENSION: "lat", LONGITUDE_DIMENSION: "lon"}
    ).sortby(["time", "lat", "lon"])
    daily = precipitation * 86400
    daily.attrs = {"units": "mm day-1"}
    monthly = daily.resample(time="MS").sum(min_count=1)
    return monthly.rename("pr").assign_attrs(units="mm month-1")


def calculate_spi(monthly, calibration_monthly):
    calibration = calibration_monthly.sel(
        time=slice(CALIBRATION_START, CALIBRATION_END)
    )
    params = standardized_index_fit_params(
        calibration,
        freq=None,
        window=SPI_SCALE_MONTHS,
        dist=SPI_DISTRIBUTION,
        method=SPI_METHOD,
        zero_inflated=True,
        fitkwargs={"floc": SPI_FLOC},
    )
    spi = standardized_precipitation_index(pr=monthly, params=params)
    spi = spi.sel(time=slice(APPLICATION_START, APPLICATION_END)).rename("spi")
    auxiliary_variables = ["number_of_zeros", "number_of_notnull", "prob_of_zero"]
    return spi.drop_vars([name for name in auxiliary_variables if name in spi.coords])

## Open and prepare the data

In [ ]:
calibration_dataset = open_precipitation(CALIBRATION_FILE)
calibration_monthly = to_monthly_precipitation(calibration_dataset)

input_dataset = open_precipitation(INPUT_FILE)
monthly_precipitation = to_monthly_precipitation(input_dataset)
if not calibration_monthly.lat.equals(monthly_precipitation.lat) or not calibration_monthly.lon.equals(monthly_precipitation.lon):
    raise ValueError("CMIP6 calibration and application data must have exactly equal latitude and longitude grids")

## Calculate and save SPI

In [ ]:
spi = calculate_spi(monthly_precipitation, calibration_monthly)
spi_dataset = spi.to_dataset()
creation_date = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
summary = (
    f"{SPI_SCALE_MONTHS}-month Standardized Precipitation Index calculated from CMIP6 experiment {EXPERIMENT} daily precipitation. "
    f"Spatial domain: lat [{float(spi.lat.min()):.2f}, {float(spi.lat.max()):.2f} deg], "
    f"lon [{float(spi.lon.min()):.2f}, {float(spi.lon.max()):.2f} deg]. "
    f"Temporal coverage: {str(spi.time.values[0])[:10]} to {str(spi.time.values[-1])[:10]}. "
    f"Calibration period: {CALIBRATION_START} to {CALIBRATION_END}."
)
spi_dataset["spi"].attrs.update(
    long_name=f"Standardized Precipitation Index ({SPI_SCALE_MONTHS}-month)",
    units="1",
    description="Positive values indicate wetter-than-normal conditions; negative values indicate drier-than-normal conditions.",
    fit_failure_interpretation="With complete precipitation input, NaN SPI values indicate that the selected probability distribution could not be fitted for that calendar month and grid cell. This can occur in very arid regions when the calibration sample contains too few positive precipitation values, producing non-finite or non-positive distribution parameters.",
    numerical_bounds="[-8.21, 8.21]",
    numerical_bounds_interpretation="When the fitted cumulative probability is numerically equal to 0 or 1, its transformation to the standard normal distribution would produce negative or positive infinity. xclim clips these values to -8.21 or 8.21. Values at these bounds represent events beyond the numerical resolution of the fitted distribution and do not necessarily indicate a failed calibration fit.",
)
spi_dataset.attrs.update(
    Conventions="CF-1.10",
    title=f"Standardized Precipitation Index for CMIP6 {EXPERIMENT}",
    summary=summary,
    creator="Marcio Cataldi <mcataldi@id.uff.br>",
    institution="Climate System Monitoring and Modeling Laboratory (LAMMOC), Universidade Federal Fluminense (UFF), Niteroi, Brazil",
    project="RiskClima",
    license="CC-BY-4.0",
    references="https://riskclima.com.br/",
    code_repository="https://github.com/lammoc-uff/cnpq-riskclima",
    processing_level="Processed data",
    source=f"CMIP6 experiment {EXPERIMENT} daily precipitation",
    keywords="spi, standardized precipitation index, CMIP6, climate projection, RiskClima",
    input_variables=PRECIPITATION_VARIABLE,
    input_frequency="daily",
    model_id=MODEL,
    experiment_id=EXPERIMENT,
    member_id=MEMBER,
    grid_label=GRID,
    calibration_period=f"{CALIBRATION_START} to {CALIBRATION_END}",
    application_period=f"{APPLICATION_START} to {APPLICATION_END}",
    calibration_method=f"{SPI_DISTRIBUTION} distribution fitted independently for each calendar month and grid cell using xclim {SPI_METHOD}, floc={SPI_FLOC}, and zero-inflated precipitation.",
    compute_backend="xarray and xclim",
    spi_scale_months=SPI_SCALE_MONTHS,
    spi_distribution=SPI_DISTRIBUTION,
    spi_fitting_method=SPI_METHOD,
    spi_floc=SPI_FLOC,
    precipitation_conversion="Daily precipitation flux in kg m-2 s-1 or equivalent mm s-1 was multiplied by 86400 to obtain mm day-1. Daily values were summed into monthly-start intervals with resample(time='MS').sum(min_count=1), producing monthly accumulated precipitation in mm month-1.",
    monthly_precipitation_units="mm month-1",
    creation_date=creation_date,
    history=f"{creation_date} Computed {SPI_SCALE_MONTHS}-month SPI using xarray and xclim.",
)
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
spi_dataset.to_netcdf(OUTPUT_FILE, engine="netcdf4", format="NETCDF4")
OUTPUT_FILE